# Runtime Context

In [1]:
"""
Runtime context is extra structured information you pass to the agent when you run it,
so the agent can use it during that execution

user message = what the user asks
runtime context = background data the system already knows for this run
"""

'\nRuntime context is extra structured information you pass to the agent when you run it,\nso the agent can use it during that execution\n\nuser message = what the user asks\nruntime context = background data the system already knows for this run\n'

In [2]:
from dotenv import load_dotenv
load_dotenv()

from pprint import pprint
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain.tools import ToolRuntime


In [3]:
# Define runtime context

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_color: str = "yellow"

In [12]:
ctx = ColourContext()
print(ctx.favourite_colour)

ctx2 = ColourContext(favourite_colour="green")
print(ctx2.favourite_colour) 

blue
green


In [4]:
# Create the agent

MODEL_NAME = "openai:gpt-5-nano"

agent = create_agent(
    model=MODEL_NAME,
    context_schema=ColourContext,
)

In [5]:
# Invoke the agent

question = HumanMessage(content="What is my favourite color?")

response = agent.invoke(
    {"messages": [question]},
    context=ColourContext()
)

pprint(response["messages"][-1].content)

('I don’t know your personal preferences yet. Want me to guess, or help you '
 'figure it out?\n'
 '\n'
 '- I can guess a color and say why people might like it.\n'
 '- I can ask you a few quick questions (warm vs. cool tones, bright vs. '
 'muted, etc.) and pick one based on your answers.\n'
 '- Or if you want a suggestion right now, blue is a common favorite for many '
 'people and pairs with lots of vibes.')


In [6]:
# Create tools that read from runtime context

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favorite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favorite colour of the user"""
    return runtime.context.least_favourite_color

In [7]:
# Create the agent

MODEL_NAME = "openai:gpt-5-nano"

agent = create_agent(
    model=MODEL_NAME,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

In [8]:
response = agent.invoke(
    {"messages": [question]},
    context=ColourContext()
)

pprint(response["messages"][-1].content)

/Users/angelo/projects/genai-portfolio/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_color='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


('Your favourite color is blue.\n'
 '\n'
 'Would you like me to change it or tell you your least favorite color as '
 'well?')


In [9]:
response = agent.invoke(
    {"messages": [question]},
    context=ColourContext(favourite_colour="green"),
)

pprint(response["messages"][-1].content)

/Users/angelo/projects/genai-portfolio/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...avourite_color='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


('Your favourite color is green. Would you like me to check your least '
 'favourite color too?')
